# Notebook 3 · Organização e exploração verificável da integração

**CIV2155 · FNDE Tipo 2 R03 · extensão incremental do Notebook 2.**

O N2 produziu um Dataset de cinco grafos, 415.175 quádruplas, 26.601 produtos e 1.652 relações espaciais. Este notebook acrescenta organização, recuperação de propriedades contextualizadas, consultas e validação. Os cinco grafos originais devem permanecer semanticamente idênticos. A geometria não é recalculada.

A análise da disciplina de Ciência de Dados motivou quatro decisões: distinguir classificação de indivíduos; conservar contexto e origem ao integrar fontes; formular perguntas antes de escolher ferramentas; comparar cada extensão com a base preservada. A extração de texto do acervo não equivale à leitura crítica integral. Inventário detalhado, transcrições, matriz de seleção e testes ficam no arquivo de pesquisa do Google Drive; o GitHub apresenta os métodos efetivamente usados.

**Fontes e adaptações:** [Ziegler e Dittrich (2007)](https://doi.org/10.1007/978-3-540-72677-7_3), contexto e níveis de integração; [Liu e Yoon (2024)](https://doi.org/10.1016/j.dss.2024.114197), orientação a objetivos, sem reprodução do sistema GoDa; [Bonduel et al. (2018), §§1.3 e 2.1](https://ceur-ws.org/Vol-2159/04paper.pdf), recursos de propriedades; [Shneiderman (1996)](https://www.cs.umd.edu/~ben/papers/Shneiderman1996eyes.pdf), navegação visual; W3C [SHACL](https://www.w3.org/TR/shacl/) e [PROV-O](https://www.w3.org/TR/prov-o/). A [documentação](../docs/Organizacao_e_Exploracao.md) explicita o que foi adaptado e os limites.

## 1. Conferir as entradas e importar as funções

Os arquivos são locais ou baixados do repositório e conferidos por SHA-256. Não se substitui silenciosamente um resultado do N2. Apenas RDFLib, IfcOpenShell e pySHACL são necessários nesta etapa; o motor Java e a geometria do N1/N2 não são repetidos. Para executar toda a sequência, produza primeiro o N2 ou utilize sua cópia publicada com o hash indicado.

In [1]:
from pathlib import Path
import os,sys,json,hashlib,zipfile,subprocess,importlib.util
from urllib.request import urlretrieve
from importlib.metadata import version
from datetime import datetime,timezone
RAIZ=Path(os.environ.get('CIV2155_RAIZ',Path.cwd())).resolve()
if not (RAIZ/'codigo').is_dir() and (RAIZ.parent/'codigo').is_dir(): RAIZ=RAIZ.parent
if not (RAIZ/'codigo').is_dir(): RAIZ=RAIZ/'CIV2155'; RAIZ.mkdir(exist_ok=True)
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
ARQUIVOS = {'codigo/organizar_dataset.py': '5519cf104189da35a4d2ab586c1ad3b63f3ea3d010d3264a52d6dae288199efb', 'codigo/Perfil_Organizacao.ttl': 'a9d10f51e06c6f3f441504920d68b661b523d4bdd7a539afa9115fee6066942c', 'codigo/Restricoes_Organizacao.ttl': '64ae08a8ac1ae6b5180bf91672970eca606a1c8683c22f48853ebc954fbe9c0f', 'referencias/organizacao/BOT_0.3.2.ttl': '0f51c7f192d5e694a9c465cbc883448c64790ffabc3c1e8d6530912c45510d73', 'requirements-exploracao.txt': 'fbfa759ae0131c91455383e9200dfe7fa97694537009aa2651e3c186720f55f6', 'dados/TIPO2-SCO-MOD_R03.IFC.zip': '9f9f0574df57606690d85cf1c875b067dd6df21e2e28a2d65bc53bfa26408efe', 'resultados/integracao/Integracao_Arquitetura_Estrutura.zip': '8fb829827ec1ad3b02285b0f24d8d8329c7c999bf3904c50533c37e3294f5959', 'resultados/integracao/Identidades_Modelos.csv': 'cac55b07044cb41814e382b41f2135b7dd1ee8ae3a6a14a21393022e36af053a', 'resultados/integracao/Integracao_Por_Ambiente.csv': '1b9355099fcbf56280abbca0aea7cfef8813e68375dd27a0771bf7955837e49e', 'codigo/consultas_exploracao/01_origem_produtos.rq': '8e5774b2cfad37e7f8a2858c6da3419cf2444699fa94a215342d985396c515cd', 'codigo/consultas_exploracao/02_ambientes_e_relacoes.rq': '431320eb4bf6c584d1680c67d581db1f1480f0a8f3fe7add0ffe9220903078f0', 'codigo/consultas_exploracao/03_cozinha_v137.rq': '2adcc5af8782f700ba200653f72c8b2699c60601c01946629f5c5ed20e354d72', 'codigo/consultas_exploracao/04_ambientes_inconclusivos.rq': '706ef1fa2e90f7e44a1a4cbe47b7956730089a297ca6c0d61c9e2c0ff8bdbbb0', 'codigo/consultas_exploracao/05_v137_propriedades_contextuais.rq': 'c380db9d1f583c531b72750c0980cb69f02d8bb4fc0fc0e0b3bdb1cea4b4c4e4', 'codigo/consultas_exploracao/06_guids_repetidos_por_modelo.rq': '50ded2016017d0e8a4f9c4307d1baa499f772dc61b30932aaf9b9d47040ede3e', 'codigo/consultas_exploracao/07_armaduras_sem_avaliacao_geometrica.rq': '9a32a33bce129760fae25a84f43f37a13eea8c8fad8326cc9ac4edafe6277976', 'codigo/consultas_exploracao/08_classes_e_niveis.rq': 'e44d18eb17887adc7f4313083719d0274c44d7aa6ad8f9e2e41270a286de5288'}
for rel,digest in ARQUIVOS.items():
    target=RAIZ/rel;target.parent.mkdir(parents=True,exist_ok=True)
    if not target.exists(): urlretrieve('https://raw.githubusercontent.com/prcgui/bim-ontology-CIV2155/main/'+rel,target)
    if sha(target)!=digest: raise ValueError('Arquivo diferente da versão deste notebook: '+rel)
deps={'rdflib':'rdflib==7.1.4','ifcopenshell':'ifcopenshell==0.8.4','pyshacl':'pyshacl==0.30.1'}
missing=[spec for mod,spec in deps.items() if importlib.util.find_spec(mod) is None or version(mod)!=spec.split('==')[1]]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
sys.path.insert(0,str(RAIZ/'codigo'))
import organizar_dataset as org
from rdflib import URIRef,Literal,RDF,RDFS
from IPython.display import display,Markdown
SAIDA=RAIZ/'resultados/organizacao';SAIDA.mkdir(parents=True,exist_ok=True)
TRABALHO=RAIZ/'.execucao/notebook3';TRABALHO.mkdir(parents=True,exist_ok=True)
INICIO=datetime.now(timezone.utc).isoformat()
print('Entradas conferidas. Início UTC:',INICIO)

Entradas conferidas. Início UTC: 2026-10-30T19:19:06.941259+00:00


## 2. Separar o que existe na fonte do que acrescentamos

`Dataset` conserva o grafo de origem de cada tripla. Uma URI identifica o produto descrito; uma classe, como `bot:Space`, classifica esse indivíduo. A coluna `classe_ifc` do catálogo preserva o tipo IFC original e não deve ser confundida com uma nova classe OWL criada no RDF.

O exemplo abaixo lê o GUID e a origem da V137. Nomes repetidos não são usados para unir identidades. Os modelos ARQ, SMT e SCO representam arquivos de origem, não uma classificação automática de sistemas construtivos. O N2 não declarava axiomas `rdfs:subClassOf`; isso não significa que BOT não possua um esquema próprio.

In [2]:
dataset,identidades,assinaturas_n2=org.carregar(RAIZ)
print('Produtos preservados:',len(identidades))
print('Quádruplas N2:',sum(len(dataset.graph(org.GRAPHS[k])) for k in assinaturas_n2))
print('Hierarquias declaradas no N2:',len(list(dataset.triples((None,RDFS.subClassOf,None)))))
v137=next(r for r in identidades if r['modelo']=='sco' and r['guid']==org.V137)
display(v137)
for p in [RDF.type,org.P.globalIdIfcRoot_attribute_simple,org.PROV.wasDerivedFrom]:
    for value in dataset.objects(URIRef(v137['uri']),p):print(p.n3(),value.n3())

Produtos preservados: 26601
Quádruplas N2: 415175
Hierarquias declaradas no N2: 0


{'modelo': 'sco',
 'guid': '0EVSuIeKOHxuDHo7zKqQBn',
 'classe_ifc': 'IfcBeam',
 'nome': 'V137',
 'uri': 'https://github.com/prcgui/bim-ontology-CIV2155/id/estrutura-sco/beam_0e7dce12-a146-11ef-8351-c87f54d1a2f1',
 'pavimento_ifc': '37aq1vEsn4dQ0oyWcjKKws'}

<http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <https://w3id.org/bot#Element>
<http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <https://pi.pauwel.be/voc/buildingelement#Beam>
<http://lbd.arch.rwth-aachen.de/props#globalIdIfcRoot_attribute_simple> "0EVSuIeKOHxuDHo7zKqQBn"
<http://www.w3.org/ns/prov#wasDerivedFrom> <https://github.com/prcgui/bim-ontology-CIV2155/id/integracao/modelo/sco>


## 3. Acrescentar um perfil em grafos separados

O esquema BOT 0.3.2 é uma referência externa, com versão e licença preservadas. O perfil próprio classifica modelos, resultados e evidências como registros informacionais derivados de `prov:Entity`. Essa escolha distingue o produto físico do registro que o descreve, sem implementar gUFO/gOCED, inventar eventos de construção ou criar `owl:sameAs`.

Os grafos `perfil` e `bot-referencia` permitem examinar níveis de abstração sem alterar os cinco grafos do N2. Não há execução de raciocinador OWL global. A hierarquia de referência é apresentada como esquema, separada da lista de indivíduos.

In [3]:
dataset.graph(org.GRAPHS['perfil']).parse(RAIZ/'codigo/Perfil_Organizacao.ttl',format='turtle')
dataset.graph(org.GRAPHS['bot-referencia']).parse(RAIZ/'referencias/organizacao/BOT_0.3.2.ttl',format='turtle')
for key in ['perfil','bot-referencia']:
    graph=dataset.graph(org.GRAPHS[key])
    print(key,len(graph),'triplas')
    for child,parent in sorted(graph.subject_objects(RDFS.subClassOf),key=lambda t:str(t[0])):
        print(' ',child,'→',parent)

perfil 56 triplas
  https://github.com/prcgui/bim-ontology-CIV2155/vocab/analise#ResultadoPorAmbiente → https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#RegistroInformacional
  https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#ConjuntoPropriedadesIFC → https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#RegistroInformacional
  https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#OcorrenciaPropriedadeIFC → https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#RegistroInformacional
  https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#RegistroInformacional → http://www.w3.org/ns/prov#Entity
  https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#UnidadeIFC → https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#RegistroInformacional
  https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#ModeloIFC → https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#RegistroInformacional
  https://github.co

## 4. Recuperar o contexto das propriedades da V137

Na extração direta L1, quatro valores aparecem sob `props:quantidade_property_simple`. O IFC identifica `Quantidade_01` a `Quantidade_04` em conjuntos distintos, acompanhadas de descrições. A extensão abaixo cria recursos para esses conjuntos e suas ocorrências de propriedades. É uma prova de conceito própria inspirada na discussão L1/L2/L3 de Bonduel et al., não uma conversão L2 completa do motor.

Para cada `IfcPropertySingleValue`, conservam-se Pset, GUID do Pset, número STEP, nome original, tipo nominal IFC, valor e unidade. Conforme a [especificação IFC4](https://standards.buildingsmart.org/IFC/RELEASE/IFC4/FINAL/HTML/schema/ifcpropertyresource/lexical/ifcpropertysinglevalue.htm), a unidade explícita prevalece; em sua ausência, consulta-se a unidade do projeto correspondente ao tipo de medida, usando `get_property_unit` do IfcOpenShell 0.8.4.

**Cuidado verificável:** o SCO declara centímetros para comprimento, mas metros quadrados, metros cúbicos e quilogramas para essas outras grandezas. Não se deve deduzir cm² e cm³ da unidade linear. Valores `IfcReal` sem unidade não recebem dimensão por suposição. A POC cobre somente Psets diretamente associados à V137 e valores simples; não cobre herança de tipos, quantidades de elementos nem todos os subtipos de propriedade.

In [4]:
config=json.loads((RAIZ/'dados/Modelos_Integracao.json').read_text(encoding='utf8'))['modelos']['sco']
ifc=TRABALHO/config['arquivo']
with zipfile.ZipFile(RAIZ/config['zip']) as z: ifc.write_bytes(z.read(config['arquivo']))
poc=org.contextualizar_v137(dataset,identidades,ifc,config['sha256_ifc'])
org.salvar(SAIDA/'Propriedades_V137.json',poc)
dataset.graph(org.GRAPHS['propriedades-contextuais']).serialize(SAIDA/'Propriedades_V137.nt',format='nt')
print(poc['conjuntos'],'conjuntos;',poc['propriedades'],'propriedades simples. Não tratados:',poc['nao_tratados'])
diretas=list(dataset.graph(org.GRAPHS['sco']).objects(URIRef(v137['uri']),org.P.quantidade_property_simple))
print('Quatro valores diretos conservados:',[x.n3() for x in diretas])
for item in poc['itens']:
    if item['nome'].startswith('Quantidade'):
        print(item['pset'],item['nome'],item['valor'],item['tipo_valor_ifc'],item['unidade'])

9 conjuntos; 34 propriedades simples. Não tratados: []
Quatro valores diretos conservados: ['"4.26151"^^<http://www.w3.org/2001/XMLSchema#decimal>', '"2.04999"^^<http://www.w3.org/2001/XMLSchema#decimal>', '"1.87392"^^<http://www.w3.org/2001/XMLSchema#decimal>', '"0.140544"^^<http://www.w3.org/2001/XMLSchema#decimal>']
AltoQi_Eberick-Item_01 Quantidade_01 0.140544 IfcVolumeMeasure {'step': 3, 'tipo': 'IfcSIUnit', 'tipo_unidade': 'VOLUMEUNIT', 'nome': 'CUBIC_METRE', 'prefixo': ''}
AltoQi_Eberick-Item_02 Quantidade_02 1.87392 IfcAreaMeasure {'step': 4, 'tipo': 'IfcSIUnit', 'tipo_unidade': 'AREAUNIT', 'nome': 'SQUARE_METRE', 'prefixo': ''}
AltoQi_Eberick-Item_03 Quantidade_03 4.26151 IfcMassMeasure {'step': 10, 'tipo': 'IfcSIUnit', 'tipo_unidade': 'MASSUNIT', 'nome': 'GRAM', 'prefixo': 'KILO'}
AltoQi_Eberick-Item_04 Quantidade_04 2.04999 IfcMassMeasure {'step': 10, 'tipo': 'IfcSIUnit', 'tipo_unidade': 'MASSUNIT', 'nome': 'GRAM', 'prefixo': 'KILO'}


C:\Users\prcgu\OneDrive\Documentos\ChatGPT\Artigo da Disciplina BIM\desenvolvimento\runtime_shacl\rdflib\plugins\serializers\nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


## 5. Validar um contrato explícito com SHACL

As restrições são próprias do estudo. A linguagem e o procedimento seguem [W3C SHACL (2017)](https://www.w3.org/TR/shacl/), sem inferência. São verificados GUID e origem dos produtos, metadados dos modelos, extremos e medidas dos registros espaciais, estado dos resultados e contexto das propriedades da POC.

A validação utiliza uma **projeção declarada**: os campos exigidos pelo contrato, tipos e proveniência de todos os 26.601 produtos; os três modelos; todos os 1.652 registros N2; os 41 resultados N1; relações BOT e dados da POC. `ctx:ProdutoIndexado` é somente um alvo local dessa projeção, não uma nova classificação física gravada nos grafos originais. Propriedades multivaloradas da fonte não são genericamente proibidas.

Passar nesse contrato não certifica alinhamento dos IFCs, projeto estrutural, completude da conversão ou consistência OWL global. Ambientes inconclusivos devem continuar sem contagem artificial. Os ensaios que injetam defeitos para conferir a detecção ficam no Drive, separados dos resultados primários.

In [5]:
validacao=org.validar(dataset,identidades,RAIZ,SAIDA)
display(validacao)
assert validacao['conforme']
assert validacao['alvos'][str(org.CTX.ProdutoIndexado)]==26601
assert validacao['alvos'][str(org.N.RelacaoEspacial)]==1652

{'conforme': True,
 'inferencia': 'none',
 'triplas_da_projecao': 200962,
 'duracao_s': 59.127,
 'alvos': {'https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#ProdutoIndexado': 26601,
  'https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#ModeloIFC': 3,
  'https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#RelacaoEspacial': 1652,
  'https://github.com/prcgui/bim-ontology-CIV2155/vocab/analise#ResultadoPorAmbiente': 41,
  'https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#ConjuntoPropriedadesIFC': 9,
  'https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#OcorrenciaPropriedadeIFC': 34},
 'alcance': 'Contrato de identidade, proveniência, resultados, vínculos e POC de propriedades. Não certifica qualidade estrutural, alinhamento, completude IFC nem consistência OWL global.'}

## 6. Responder perguntas antes de visualizar

As consultas partem de necessidades concretas: produtos e origem; relações por ambiente; evidência da V137 na cozinha; ambientes inconclusivos; contexto das propriedades; GUID repetido dentro do mesmo modelo; armaduras fora da avaliação geométrica; hierarquias dos esquemas.

A orientação a objetivos dialoga com Liu e Yoon (2024), sem reproduzir GoDa, correspondência automática de esquemas, NLP ou aprendizado de máquina. As consultas são arquivos `.rq` editáveis. Resultados ausentes permanecem ausentes. A contagem de armaduras sem registro espacial indica cobertura do procedimento, não ausência de relação física.

In [6]:
consultas=org.consultar(dataset,RAIZ,SAIDA)
for consulta in consultas:
    print(consulta['id'],consulta['linhas'],'linhas')
display(consultas[2]['resultados'])
display(consultas[3]['resultados'])
display(consultas[4]['resultados'])
assert consultas[0]['linhas']==3
assert consultas[2]['linhas']==1
assert consultas[3]['linhas']==3
assert consultas[4]['linhas']==poc['propriedades']
assert consultas[5]['linhas']==0
assert consultas[6]['resultados'][0]['armaduras_sem_registro_espacial']=='17755'

CONSULTA 01_origem_produtos


CONSULTA 02_ambientes_e_relacoes


CONSULTA 03_cozinha_v137


CONSULTA 04_ambientes_inconclusivos


CONSULTA 05_v137_propriedades_contextuais


...

CONSULTA 08_classes_e_niveis


01_origem_produtos 3 linhas
02_ambientes_e_relacoes 54 linhas
03_cozinha_v137 1 linhas
04_ambientes_inconclusivos 3 linhas
05_v137_propriedades_contextuais 34 linhas
06_guids_repetidos_por_modelo 0 linhas
07_armaduras_sem_avaliacao_geometrica 1 linhas
08_classes_e_niveis 11 linhas


[{'ambiente': 'https://github.com/prcgui/bim-ontology-CIV2155/id/notebook1-v03/space_de3ed29d-12e1-45f6-af99-d3219f5ba16f',
  'elemento': 'https://github.com/prcgui/bim-ontology-CIV2155/id/estrutura-sco/beam_0e7dce12-a146-11ef-8351-c87f54d1a2f1',
  'predicado': 'https://w3id.org/bot#adjacentElement',
  'estado': 'adjacente',
  'areaM2': '0.4674837228519026',
  'volumeM3': '0.0',
  'modelo': 'sco',
  'grafo': 'https://github.com/prcgui/bim-ontology-CIV2155/grafo/sco',
  'sha256': 'bde759756938510c56112b9544aa2a26aca08b6a229008de85c5144d2961ea93'}]

[{'ambiente': 'https://github.com/prcgui/bim-ontology-CIV2155/id/notebook1-v03/space_5e17dff5-4488-4291-abba-92395059a4d4',
  'nome': 'AMAMENTAÇÃO',
  'motivo': 'Ambiente não forma uma única casca fechada; sem reparação automática; recuperação controlada sem candidato aceito',
  'quantidade': None},
 {'ambiente': 'https://github.com/prcgui/bim-ontology-CIV2155/id/notebook1-v03/space_bbfa3934-1b3d-4529-a9c4-10714c19a7a4',
  'nome': 'INFANTIL 1',
  'motivo': 'Ambiente não forma uma única casca fechada; sem reparação automática; recuperação controlada sem candidato aceito',
  'quantidade': None},
 {'ambiente': 'https://github.com/prcgui/bim-ontology-CIV2155/id/notebook1-v03/space_8600f390-0e41-48e4-b329-913a579a8477',
  'nome': 'INFANTIL 5',
  'motivo': 'Ambiente não forma uma única casca fechada; sem reparação automática; recuperação controlada sem candidato aceito',
  'quantidade': None}]

[{'pset': 'AltoQi_Eberick-Item_01',
  'nome': 'Descrição_01',
  'tipo': 'IfcText',
  'valor': 'Concreto - C-30 - Abatimento 12 cm',
  'unidade': None,
  'prefixo': None,
  'origemUnidade': 'nao_declarada_ou_nao_dimensional',
  'step': '430382'},
 {'pset': 'AltoQi_Eberick-Item_01',
  'nome': 'Quantidade_01',
  'tipo': 'IfcVolumeMeasure',
  'valor': '0.140544',
  'unidade': 'CUBIC_METRE',
  'prefixo': '',
  'origemUnidade': 'unidades_do_projeto_por_tipo_da_medida',
  'step': '430459'},
 {'pset': 'AltoQi_Eberick-Item_01',
  'nome': 'Status_01',
  'tipo': 'IfcText',
  'valor': 'Dimensionado',
  'unidade': None,
  'prefixo': None,
  'origemUnidade': 'nao_declarada_ou_nao_dimensional',
  'step': '432812'},
 {'pset': 'AltoQi_Eberick-Item_02',
  'nome': 'Descrição_02',
  'tipo': 'IfcText',
  'valor': 'Forma - Estrutura - Concreto',
  'unidade': None,
  'prefixo': None,
  'origemUnidade': 'nao_declarada_ou_nao_dimensional',
  'step': '430387'},
 {'pset': 'AltoQi_Eberick-Item_02',
  'nome': 'Qua

## 7. Gerar o índice para exploração e conservar o Dataset completo

O explorador usa visão geral, filtros, detalhes, relações, histórico e extração de recortes, adaptando [Shneiderman (1996), §§3 e 4](https://www.cs.umd.edu/~ben/papers/Shneiderman1996eyes.pdf). A navegação inclui todos os produtos identificados dos três arquivos; o desenho mostra apenas uma vizinhança por vez. A classe IFC, os tipos RDF, o grafo e o GUID são distinguidos. A página de classes mostra os esquemas separadamente dos indivíduos.

O índice visual contém triplas diretas dos produtos e registros selecionados; literais extensos de geometria e recursos auxiliares permanecem no Dataset integral. A exportação do foco conserva termos RDF e grafos em N-Quads, acompanhada do estado dos filtros. Os filtros afetam a exibição; o recorte exportado inclui o contexto do foco descrito na interface.

O arquivo do N2, cada um de seus cinco grafos e todos os grafos reabertos são conferidos. A nova atividade PROV-O representa esta execução computacional, não eventos reais da obra. Não é necessário modificar a geometria dos notebooks anteriores.

In [7]:
indice=org.criar_indice(dataset,identidades,RAIZ,SAIDA,poc,consultas)
resumo=org.finalizar(dataset,assinaturas_n2,RAIZ,SAIDA,INICIO,
    {'produtos':len(identidades),'psets_poc':poc['conjuntos'],'propriedades_poc':poc['propriedades'],
     'relacoes_n2':1652,'pares_n1':101,'shacl_conforme':validacao['conforme']})
display(indice)
display(resumo)
assert resumo['cinco_grafos_n2_inalterados'] and resumo['reabertura_sem_alteracao_de_termos']

{'produtos': 26601,
 'ambientes': 41,
 'quadruplas_no_indice': 394067,
 'literais_extensos_omitidos_por_predicado': {},
 'sha256_n2': '8fb829827ec1ad3b02285b0f24d8d8329c7c999bf3904c50533c37e3294f5959',
 'escopo': 'Triplas diretas dos produtos e registros selecionados. Geometria extensa e recursos auxiliares permanecem no Dataset integral.'}

{'produtos': 26601,
 'psets_poc': 9,
 'propriedades_poc': 34,
 'relacoes_n2': 1652,
 'pares_n1': 101,
 'shacl_conforme': True,
 'grafos': {'arquitetura': 80234,
  'smt': 84933,
  'sco': 205194,
  'relacoes': 18194,
  'proveniencia': 26620,
  'perfil': 56,
  'bot-referencia': 957,
  'propriedades-contextuais': 406,
  'execucao-n3': 9},
 'quadruplas': 416603,
 'cinco_grafos_n2_inalterados': True,
 'reabertura_sem_alteracao_de_termos': True,
 'sha256_entrada_n2': '8fb829827ec1ad3b02285b0f24d8d8329c7c999bf3904c50533c37e3294f5959',
 'sha256_saida': 'bc13ed84f73be01875cc33fd932285a68adc2043c94d2d9127bf898fa9c60ecf',
 'inicio_utc': '2026-10-30T19:19:06.941259+00:00',
 'fim_utc': '2026-10-30T19:22:29.664819+00:00'}

## 8. Interpretar o ganho e seus limites

**Ganho demonstrável:** os mesmos produtos e relações podem ser consultados por origem; os registros são distinguidos dos produtos descritos; a V137 conserva contexto e unidade das propriedades; o contrato SHACL é executável; o explorador reúne fontes e evidencia dados ausentes. A comparação preserva a base N1/N2. O explorador anterior do N1 já tinha navegação organizada e permanece acessível; o ganho aqui é ampliar a cobertura e tornar a integração verificável.

**Não demonstrado:** aumento de acurácia geométrica, alinhamento independente, identidade física entre modelos, adequação estrutural, harmonização global de propriedades, precisão de aprendizagem de máquina ou ganho de usabilidade medido com participantes. As três salas inconclusivas e os elementos fora da avaliação geométrica permanecem limitações. Reorganizar visualmente um grafo não resolve automaticamente conflitos semânticos.

**Artefatos:** [explorador publicado](https://prcgui.github.io/bim-ontology-CIV2155/), [resultados primários](../resultados/organizacao/), [referências e adaptações](../referencias/organizacao/README.md). O manuscrito atual não foi localizado nas pastas de redação verificadas. A [contribuição textual delimitada](../docs/Contribuicoes_para_Artigo.md) pode ser incorporada posteriormente; não se afirma revisão de um artigo inexistente nas fontes acessadas nem conferência de suas 15 páginas.